# 10주차 복습 과제: 하루스테이 예약 챗봇
## 🏨 시나리오: 숙박 예약 서비스 "하루스테이"

```
여러분은 숙박 예약 스타트업 '하루스테이'의 주니어 개발자입니다.
대표의 요청: "예약 상담 챗봇을 만들어주세요. 예약자 이름이랑 예약번호는
           한 번만 입력하면 대화 내내 기억해야 하고,
           문의가 길어져도 느려지면 안 됩니다."
```

실습에서 만든 위니브마켓 상담봇과 **구조가 동일**합니다.
빈칸(`# ??`)을 채우면서 각 코드가 왜 그렇게 생겼는지 다시 떠올려보세요.

| 실습 매핑 | 위니브마켓(실습) | 하루스테이(과제) |
| --- | --- | --- |
| 페르소나 | 위니봇 | 하루봇 |
| 식별 정보 | 고객 이름 + 주문번호 | 예약자 이름 + 예약번호 |
| 문의 예시 | 배송 지연 | 체크인 시간 변경 |

| Part | 내용 | 배점 |
| --- | --- | --- |
| A-1 | MessagesPlaceholder로 프롬프트 구성 | 15점 |
| A-2 | RunnableWithMessageHistory로 메모리 연결 | 15점 |
| A-3 | session_id로 다중 예약자 분리 | 20점 |
| A-4 | 요약 체인 직접 구성 | 20점 |
| B | 직접 구현 (옵션 선택) | 30점 |

> ⚠️ **시작 전 체크**: 아래 `0️⃣` 셀에서 API Key를 입력하세요.

---
## 0️⃣ 환경 설정

In [1]:
# 패키지 설치 (Colab 최초 1회)
!pip install -q langchain langchain-google-genai
print('✅ 설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.1 MB/s eta 0:00:00
✅ 설치 완료


In [2]:
import os

# ──────────────────────────────────────────────
# ✏️  여기에 API Key 입력 (Google AI Studio 발급)
os.environ['GOOGLE_API_KEY'] = 'API_KEY'
# ──────────────────────────────────────────────

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.3,
)
print('✅ LLM 준비 완료 — 하루스테이 예약봇 개발 시작!')

✅ LLM 준비 완료 — 하루스테이 예약봇 개발 시작!


---
## 📋 Part A — 빈칸 채우기 (70점)

`# ??` 표시가 있는 부분을 채워주세요. 각 문제는 마지막에 `assert`로 자동 채점됩니다.

### A-1. MessagesPlaceholder로 프롬프트 구성 (15점)

**힌트**: 실습 셀 3️⃣를 참고하세요. `MessagesPlaceholder`는 대화 이력이 들어갈 자리를 예약합니다.
`variable_name`은 나중에 `RunnableWithMessageHistory`의 `history_messages_key`와 이름이 같아야 합니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 하루스테이 예약 상담원입니다. 친절하게 응대하세요.'),
    # ?? : 대화 이력이 들어갈 자리를 예약하는 클래스를 사용하세요.
    #      variable_name은 'history'로 지정합니다.
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

chain = prompt | llm

# ── 자동 채점 (수정하지 마세요) ──
placeholder_found = any(
    type(m).__name__ == 'MessagesPlaceholder' and getattr(m, 'variable_name', None) == 'history'
    for m in prompt.messages
)
assert placeholder_found, '❌ MessagesPlaceholder(variable_name="history")가 프롬프트에 없습니다.'
print('✅ A-1 통과!')

✅ A-1 통과!


### A-2. RunnableWithMessageHistory로 메모리 연결 (15점)

**힌트**: 실습 셀 4️⃣를 참고하세요. `input_messages_key`와 `history_messages_key`가
각각 어떤 역할을 하는지 떠올려보세요.

In [4]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',    # ?? : 현재 사용자 입력이 들어갈 키 이름 ('{input}'과 일치해야 함)
    history_messages_key='history',  # ?? : 대화 이력이 들어갈 키 이름 (A-1의 variable_name과 일치해야 함)
)

# ── 자동 채점 (수정하지 마세요) ──
config_test = {'configurable': {'session_id': 'test_a2'}}
r1 = chain_with_history.invoke({'input': '안녕하세요. 저는 박서연이고 예약번호는 HS-0001입니다.'}, config=config_test)
r2 = chain_with_history.invoke({'input': '제 이름이 뭐였죠?'}, config=config_test)

assert '서연' in r2.content, '❌ 메모리가 작동하지 않습니다. input_messages_key/history_messages_key를 다시 확인하세요.'
print('✅ A-2 통과!')
print('   [확인 응답]', r2.content[:60])

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


✅ A-2 통과!
   [확인 응답] 네, 박서연 고객님이십니다! 😊

혹시 다른 문의사항이 있으실까요? 편하게 말씀해주세요.


### A-3. session_id로 다중 예약자 분리 (20점)

**힌트**: 실습 셀 1️⃣1️⃣~1️⃣2️⃣를 참고하세요. `session_id`가 다르면 완전히 독립된 대화가 유지됩니다.
두 예약자(GUEST-A, GUEST-B)가 동시에 상담해도 정보가 섞이지 않아야 합니다.

In [5]:
def consult(guest_id: str, message: str) -> str:
    """하루봇과 상담하는 함수. guest_id로 예약자를 구분합니다."""
    response = chain_with_history.invoke(
        {'input': message},
        # ?? : config 딕셔너리를 완성하세요.
        #      'configurable' 안에 'session_id' 키로 guest_id를 전달해야 합니다.
        config={'configurable': {'session_id': guest_id}},
    )
    return response.content

# 예약자 A
consult('GUEST-A', '안녕하세요. 박서연이고 예약번호는 HS-20241220-0007이에요.')
consult('GUEST-A', '체크인을 오후 3시로 변경하고 싶어요.')

# 예약자 B (A와 동시)
consult('GUEST-B', '안녕하세요. 최도윤입니다. 예약번호 HS-20241221-0015 취소하고 싶어요.')

# ── 자동 채점 (수정하지 마세요) ──
ra = consult('GUEST-A', '제 이름이랑 예약번호 확인해주세요.')
rb = consult('GUEST-B', '제 이름이랑 예약번호 확인해주세요.')

print('[GUEST-A 응답]', ra)
print('[GUEST-B 응답]', rb)
print()

assert '서연' in ra, '❌ GUEST-A 응답에 본인 이름이 없습니다.'
assert '도윤' in rb, '❌ GUEST-B 응답에 본인 이름이 없습니다.'

# 참고: 위 두 assert는 각자 '본인' 정보를 정확히 기억하는지 확인합니다.
# 두 응답을 직접 눈으로도 비교해보세요 — 서로의 예약번호가 섞여있지 않은지 확인하는 것이
# session_id 분리가 제대로 동작했는지 보는 가장 확실한 방법입니다.
print('✅ A-3 통과! 두 예약자의 대화가 정확히 분리되었습니다.')

[GUEST-A 응답] 네, 박서연 고객님. 다시 한번 확인해 드리겠습니다.

고객님의 성함은 **박서연** 님이시며, 예약번호는 **HS-20241220-0007**이 맞습니다.

혹시 다른 문의사항은 없으실까요?
[GUEST-B 응답] 네, 최도윤 고객님. 고객님의 성함과 예약번호 HS-20241221-0015는 제가 확인하고 있습니다.

다만, 고객님의 소중한 개인 정보 보호와 예약의 안전한 처리를 위해, **본인 확인 절차**가 필요합니다.

예약 시 등록하신 **휴대폰 번호 뒷 4자리** 또는 **이메일 주소**를 다시 한번 알려주시면 감사하겠습니다.

이 정보를 통해 본인 확인이 완료되면, 즉시 취소 절차를 진행해 드리겠습니다. 양해 부탁드립니다. 😊

✅ A-3 통과! 두 예약자의 대화가 정확히 분리되었습니다.


### A-4. 요약 체인 직접 구성 (20점)

**힌트**: 실습 셀 7️⃣를 참고하세요. `RunnableWithMessageHistory`는 이력을 저장만 할 뿐
요약 기능은 없습니다. '기존 요약 + 새 대화 → 새 요약'을 만드는 체인을 직접 구성해야 합니다.

In [6]:
from langchain_core.output_parsers import StrOutputParser

summary_prompt = ChatPromptTemplate.from_messages([
    ('system', '아래 예약 상담 요약과 새로운 대화를 한 문단으로 압축하세요. 예약자 이름, 예약번호, 문의 핵심은 반드시 남기세요.'),
    # ?? : human 메시지 템플릿을 작성하세요.
    #      {summary}, {user_input}, {ai_output} 세 변수를 모두 포함해야 합니다.
    #      (힌트: '[기존 요약]\n{summary}\n\n[새 대화]\n고객: {user_input}\n상담원: {ai_output}' 형태)
    ('human', '[기존 요약]\n{summary}\n\n[새 대화]\n고객: {user_input}\n상담원: {ai_output}'),
])

summary_chain = summary_prompt | llm | StrOutputParser()

def update_summary(summary: str, user_input: str, ai_output: str) -> str:
    return summary_chain.invoke({
        'summary': summary if summary else '(아직 대화 없음)',
        'user_input': user_input,
        'ai_output': ai_output,
    })

# ── 자동 채점 (수정하지 마세요) ──
test_summary = ''
test_summary = update_summary(test_summary, '저는 최도윤이고 예약번호 HS-0015입니다.', '안녕하세요, 최도윤 고객님!')
test_summary = update_summary(test_summary, '체크아웃을 1시간 늦추고 싶어요.', '네, 확인해 드리겠습니다.')

assert len(test_summary) > 0, '❌ 요약 결과가 비어 있습니다.'
assert '도윤' in test_summary or 'HS-0015' in test_summary, '❌ 핵심 정보(이름 또는 예약번호)가 요약에 남아있지 않습니다.'
print('✅ A-4 통과!')
print('   [생성된 요약]', test_summary)

✅ A-4 통과!
   [생성된 요약] 예약자 최도윤 고객(예약번호 HS-0015)은 고객 정보 확인 후, 체크아웃 시간을 1시간 늦추고 싶다고 문의했으며, 상담원은 이에 대해 확인 중입니다.


---
## 🚀 Part B — 직접 구현 (30점)

빈칸 채우기 없이, 아래 두 옵션 중 **하나를 선택**해 직접 구현하세요.
코드 스타일은 자유롭게 작성해도 됩니다.

### 옵션 선택

**옵션 1 — 예약 변경/취소 응대 보완**
하루봇의 시스템 프롬프트를 보완해서, 고객이 "변경"이나 "취소"를 언급하면
반드시 예약번호를 재확인하고 처리 절차를 안내하도록 만드세요.

**옵션 2 — 3명 이상 동시 예약자 처리**
`session_id`를 3개 이상 사용해 동시 상담 시나리오를 시뮬레이션하고,
전체 세션 현황(예약자별 대화 턴수, 최근 메시지)을 출력하는 코드를 작성하세요.

---

**선택한 옵션을 적어주세요:** `옵션 1`

In [7]:
# Part B 구현 공간
# 아래에 자유롭게 코드를 작성하세요.
# 시스템 프롬프트를 새로 정의하거나, 기존 chain_with_history를 재사용해도 됩니다.

# 예시 골격 (옵션 1을 고른 경우)
# HARUBOT_PROMPT_V2 = '''당신은 하루스테이의 예약 상담원 하루봇입니다.
#
# 핵심 규칙:
# 1. ...
# 2. 고객이 변경이나 취소를 언급하면 예약번호를 다시 확인하고 절차를 안내하세요.
# '''

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. 예약 변경/취소 규칙이 보완된 새 시스템 프롬프트 정의
HARUBOT_PROMPT_V2 = """당신은 하루스테이의 AI 예약 상담원 하루봇입니다.

핵심 규칙:
1. 고객이 이름을 알려주면 이후 대화에서 반드시 고객님이라는 호칭과 함께 이름을 불러주세요.
2. 고객이 예약 '변경'이나 '취소'를 언급하면, 반드시 예약번호를 다시 확인(또는 요청)하고 아래 절차를 안내하세요.
   - 예약 변경 절차: 예약번호 확인 -> 변경 원하는 날짜/시간 접수 -> 가능 여부 조회 및 변경 완료 안내
   - 예약 취소 절차: 예약번호 확인 -> 취소 사유 접수 -> 환불 규정 안내 및 취소 접수 완료 안내
3. 답변은 2~3문장으로 간결하게, 항상 친절하고 정중하게 응대하세요."""

# 2. 새로운 프롬프트 템플릿 및 체인 구성
harubot_prompt_v2 = ChatPromptTemplate.from_messages([
    ('system', HARUBOT_PROMPT_V2),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

harubot_chain_v2 = harubot_prompt_v2 | llm

# 3. 메모리가 연결된 최종 V2 체인 생성 (기존 get_session_history 함수 재사용)
harubot_with_history_v2 = RunnableWithMessageHistory(
    harubot_chain_v2,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)

# 4. 편의를 위한 테스트용 상담 함수 정의
def consult_v2(guest_id: str, message: str):
    response = harubot_with_history_v2.invoke(
        {'input': message},
        config={'configurable': {'session_id': guest_id}}
    )
    print(f"[고객] {message}")
    print(f"[하루봇] {response.content}\n")





/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
# Part B 동작 확인
# 구현한 기능이 실제로 작동하는지 보여주는 테스트 코드를 작성하고 실행하세요.
# (출력 결과가 채점에 포함됩니다)

# --- 테스트 시나리오 1: 예약 변경 요청 ---
print("=== 시나리오 1: GUEST-C 예약 변경 테스트 ===")
config_c = {'configurable': {'session_id': 'GUEST-C'}}

consult_v2('GUEST-C', '안녕하세요, 저 박지민인데 예약 변경하고 싶어서요.')
consult_v2('GUEST-C', '제 예약번호는 HS-2026-9999입니다. 일정을 하루 뒤로 미루고 싶어요.')


# --- 테스트 시나리오 2: 예약 취소 요청 ---
print("=== 시나리오 2: GUEST-D 예약 취소 테스트 ===")
config_d = {'configurable': {'session_id': 'GUEST-D'}}

consult_v2('GUEST-D', '안녕하세요. 개인 사정이 생겨서 예약을 취소해야 할 것 같아요.')





=== 시나리오 1: GUEST-C 예약 변경 테스트 ===
[고객] 안녕하세요, 저 박지민인데 예약 변경하고 싶어서요.
[하루봇] 안녕하세요, 박지민 고객님! 예약 변경을 도와드리겠습니다.
먼저 예약번호를 알려주시면 빠르게 확인해 드릴 수 있습니다. 예약번호를 말씀해 주시겠어요?

[고객] 제 예약번호는 HS-2026-9999입니다. 일정을 하루 뒤로 미루고 싶어요.
[하루봇] 네, 박지민 고객님, 예약번호 HS-2026-9999 확인했습니다.
일정을 하루 뒤로 변경 원하시는군요. 해당 날짜에 객실 이용이 가능한지 확인 후 다시 안내해 드리겠습니다. 잠시만 기다려 주세요!

=== 시나리오 2: GUEST-D 예약 취소 테스트 ===
[고객] 안녕하세요. 개인 사정이 생겨서 예약을 취소해야 할 것 같아요.
[하루봇] 안녕하세요, 하루봇입니다. 예약 취소로 불편을 드려 죄송합니다.

예약 취소를 도와드리기 위해 예약번호를 알려주시겠어요? 예약번호를 확인한 후, 취소 절차와 환불 규정에 대해 자세히 안내해 드리겠습니다.



---
## 📝 회고 (필수 작성)

In [9]:
review = """
[과제를 하면서 느낀 점]
- 실습과 다르게 헷갈렸던 부분:
  A-4 문제에서 요약 체인의 human 메시지 템플릿을 작성할 때, LangChain 프롬프트 내부의 변수명({summary}, {user_input}, {ai_output})과 문자열 구조를 일치시키는 부분이 실습 코드와 매칭하면서 순간 헷갈렸습니다. 하지만 힌트를 보고 차근차근 구조를 잡아가며 이해할 수 있었습니다.

- 가장 이해가 잘 된 개념:
  MessagesPlaceholder와 RunnableWithMessageHistory의 연동 원리입니다. 프롬프트에 공간(Placeholder)을 비워두고, 래퍼가 세션 ID별로 InMemoryChatMessageHistory에서 이력을 꺼내와 그 자리에 자동으로 채워주는 흐름이 무척 직관적이어서 이해가 잘 되었습니다.

- Part B에서 선택한 옵션과 이유:
  옵션 1(예약 변경/취소 응대 보완)을 선택했습니다. 실제 서비스 환경에서는 예약의 변경이나 취소가 민감한 보안 작업이기 때문에, 시스템 프롬프트(Persona) 차원에서 이를 엄격하게 규제하고 특정 절차(예약번호 재확인 및 단계별 안내)를 강제하는 규칙을 직접 설계해보고 싶었기 때문입니다.
"""
print(review)


[과제를 하면서 느낀 점]
- 실습과 다르게 헷갈렸던 부분: 
  A-4 문제에서 요약 체인의 human 메시지 템플릿을 작성할 때, LangChain 프롬프트 내부의 변수명({summary}, {user_input}, {ai_output})과 문자열 구조를 일치시키는 부분이 실습 코드와 매칭하면서 순간 헷갈렸습니다. 하지만 힌트를 보고 차근차근 구조를 잡아가며 이해할 수 있었습니다.

- 가장 이해가 잘 된 개념: 
  MessagesPlaceholder와 RunnableWithMessageHistory의 연동 원리입니다. 프롬프트에 공간(Placeholder)을 비워두고, 래퍼가 세션 ID별로 InMemoryChatMessageHistory에서 이력을 꺼내와 그 자리에 자동으로 채워주는 흐름이 무척 직관적이어서 이해가 잘 되었습니다.

- Part B에서 선택한 옵션과 이유: 
  옵션 1(예약 변경/취소 응대 보완)을 선택했습니다. 실제 서비스 환경에서는 예약의 변경이나 취소가 민감한 보안 작업이기 때문에, 시스템 프롬프트(Persona) 차원에서 이를 엄격하게 규제하고 특정 절차(예약번호 재확인 및 단계별 안내)를 강제하는 규칙을 직접 설계해보고 싶었기 때문입니다.



---
## ✅ 제출 전 체크리스트

- [ ] API Key를 `YOUR_API_KEY`로 다시 바꿨나요?
- [ ] A-1 ~ A-4 모두 `✅ 통과!` 메시지를 확인했나요?
- [ ] Part B 옵션 중 하나를 선택해 구현하고 테스트 출력을 남겼나요?
- [ ] 회고를 작성했나요?
- [ ] 파일명을 `10주차_복습과제_이름.ipynb` 형식으로 바꿨나요?